Przygotowanie danych do modelu

In [16]:
import pandas as pd
from pathlib import Path

# --- Ścieżki względem bieżącego pliku notebooka ---
NOTEBOOK_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
DATA_RAW = NOTEBOOK_DIR.parent / "data" / "house_plants_with_toxicity.csv"
EXPORT_DIR = NOTEBOOK_DIR.parent / "data" / "exports" / "eda_temp"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def save_csv(df, name):
    path = EXPORT_DIR / name
    df.to_csv(path, index=False)
    print(f"💾 Zapisano: {path}")

def load_csv(name):
    path = EXPORT_DIR / name
    print(f"📂 Wczytywanie: {path}")
    return pd.read_csv(path)

# --- Wczytanie surowych danych ---
print("📍 Notebook dir:", NOTEBOOK_DIR)
print("📍 CSV path:", DATA_RAW)
df = pd.read_csv(DATA_RAW, low_memory=False)
print("✅ Dane wczytane:", df.shape)
df.head()


📍 Notebook dir: c:\Users\annas\projekty_infoshare\projekt ML\notebooks
📍 CSV path: c:\Users\annas\projekty_infoshare\projekt ML\data\house_plants_with_toxicity.csv
✅ Dane wczytane: (209, 20)


,id,latin,family,common,category,origin,climate,ideallight,toleratedlight,watering,insects,diseases,use,tempmax.celsius,tempmax.fahrenheit,tempmin.celsius,tempmin.fahrenheit,latin_norm,toxicity_animals,toxicity_humans
0,0,Aeschynanthus lobianus,Gesneriaceae,Lipstick,Hanging,Java,Tropical,Bright light,Direct sunlight,Keep moist between watering. Can be a bit dry ...,Mealy bug; Aphid; Thrips,NaN,Hanging; Flower; Tertiary,32,89.6,14,57.2,Aeschynanthus lobianus,nietoksyczna,nietoksyczna
1,1,Adiantum raddianum,Polypodiaceae,Maindenhair; Delta maidenhair,Fern,Brazil,Tropical,Bright light,Diffused,Keep moist between watering. Must not be dry b...,Mealy bug; Aphid; Snail,Gray mold,Potted plant; Ground cover; Table top,30,86.0,12,53.6,Adiantum raddianum,nietoksyczna,nietoksyczna
2,2,Aechmea fatsiata,Bromeliaceae,Silver vase,Bromeliad,Brazil,Tropical humid,Bright light,Diffused,Water when soil is half dry. Change water in t...,NaN,NaN,Flower; Table top; Tertiary,30,20.0,12,53.6,Aechmea fatsiata,nietoksyczna,może powodować lekkie podrażnienia
3,3,Agave angustilolia Marginata,Amaryllidaceae,Variegated Carabbean Agave; Century plant,Cactus And Succulent,India,Tropical,6 or more hours of direct sunlight per day.,Direct sunlight.,Water only when the soil is dry. Must be dry b...,Scale; Mealy bug,NaN,Potted plant; Primary; Secondary,35,95.0,5,41.0,Agave angustilolia Marginata,"toksyczna (podrażnienia, wymioty)","toksyczna (sok drażni skórę, oczy)"
4,4,Aechmea ramosa,Bromeliaceae,Coral berry,Bromeliad,Brazil,Subtropical,Bright light,Diffused,Water when soil is half dry. Change water in t...,NaN,NaN,Flower; Table top; Primary,30,20.0,12,53.6,Aechmea ramosa,nietoksyczna,może powodować lekkie podrażnienia


In [17]:
# sprawdzanie typow danych

print(df.dtypes)        

id                      int64
latin                  object
family                 object
common                 object
category               object
origin                 object
climate                object
ideallight             object
toleratedlight         object
watering               object
insects                object
diseases               object
use                    object
tempmax.celsius         int64
tempmax.fahrenheit    float64
tempmin.celsius         int64
tempmin.fahrenheit    float64
latin_norm             object
toxicity_animals       object
toxicity_humans        object
dtype: object


In [18]:
#sprawdzanie braków i duplikatów

# liczba brakujących wartości w każdej kolumnie
missing = df.isna().sum().sort_values(ascending=False)
print("🔎 Brakujące wartości w kolumnach:")
print(missing[missing > 0])  

# liczba duplikatów (identycznych wierszy)
duplicates = df.duplicated().sum()
print(f"\n🔎 Liczba duplikatów w całej tabeli: {duplicates}")

# podgląd kilku duplikatów (jeśli istnieją)
if duplicates > 0:
    print("\nPrzykładowe duplikaty:")
    display(df[df.duplicated()].head())


🔎 Brakujące wartości w kolumnach:
diseases    147
insects      37
common       15
dtype: int64

🔎 Liczba duplikatów w całej tabeli: 0


In [19]:
# podejrzenie ile roslin ma identyczne cechy a rózni się nazwą zwyczajową

import itertools

# kontrola wymaganych kolumn
required = {"common", "id"}
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise KeyError(f"Brakuje kolumn: {missing_required}")

# kolumny bazowe (wszystkie poza 'common' i 'id')
base_cols = [c for c in df.columns if c not in ["common", "id"]]

g = df.groupby(base_cols, dropna=False)
sizes = g.size()

# bierzemy tylko te grupy, gdzie po kolumnach bazowych jest więcej niż 1 wiersz
selected_idx = []
group_count = 0
rows_in_selected = 0
max_group_size = 0

for key, sub in g:
    if len(sub) > 1:
        # sprawdź, czy w tej grupie występuje JAKAŚ różnica w 'common' lub 'id'
        var_common = sub["common"].nunique(dropna=False) > 1
        var_id = sub["id"].nunique(dropna=False) > 1
        if var_common or var_id:
            group_count += 1
            rows_in_selected += len(sub)
            max_group_size = max(max_group_size, len(sub))
            selected_idx.append(sub.index)

total_rows = len(df)
print("🔎 Grupy (inne kolumny identyczne) z różnicą tylko w 'common' i/lub 'id':", group_count)
print(f"🔎 Wiersze należące do tych grup: {rows_in_selected}  ({rows_in_selected/total_rows:.2%} zbioru)")
if group_count:
    print(f"🔎 Średnia liczebność grupy: {rows_in_selected/group_count:.2f}")
    print(f"🔎 Maksymalna liczebność grupy: {max_group_size}")


🔎 Grupy (inne kolumny identyczne) z różnicą tylko w 'common' i/lub 'id': 13
🔎 Wiersze należące do tych grup: 45  (21.53% zbioru)
🔎 Średnia liczebność grupy: 3.46
🔎 Maksymalna liczebność grupy: 13


In [22]:
# usunięcie duplikatów z kolumny latin

df = df.drop_duplicates(subset=["latin"], keep="first") 

#zapisanie zmian do pliku CSV   

df.to_csv(EXPORT_DIR / "plants_clean.csv", index=False)
df = pd.read_csv(EXPORT_DIR / "plants_clean.csv")



In [23]:
# Usuwamy niepotrzebne kolumny 
cols_to_drop = ["id", "common", "diseases"]

df = df.drop(columns=cols_to_drop, errors="ignore")

print("✅ Usunięto kolumny:", cols_to_drop)
print("📐 Aktualny rozmiar:", df.shape)
print("🔎 Kolumny pozostałe:", df.columns.tolist())

df.to_csv(EXPORT_DIR / "plants_clean1.csv", index=False)
df = pd.read_csv(EXPORT_DIR / "plants_clean1.csv")



✅ Usunięto kolumny: ['id', 'common', 'diseases']
📐 Aktualny rozmiar: (159, 17)
🔎 Kolumny pozostałe: ['latin', 'family', 'category', 'origin', 'climate', 'ideallight', 'toleratedlight', 'watering', 'insects', 'use', 'tempmax.celsius', 'tempmax.fahrenheit', 'tempmin.celsius', 'tempmin.fahrenheit', 'latin_norm', 'toxicity_animals', 'toxicity_humans']


In [24]:

print("🔎 Lista wszystkich kolumn w dataframe:")
for col in df.columns:
    print(f"- {col}")


🔎 Lista wszystkich kolumn w dataframe:
- latin
- family
- category
- origin
- climate
- ideallight
- toleratedlight
- watering
- insects
- use
- tempmax.celsius
- tempmax.fahrenheit
- tempmin.celsius
- tempmin.fahrenheit
- latin_norm
- toxicity_animals
- toxicity_humans


In [25]:
# po analizie tresci kolumn zadecydowano o usunięciu nastepujących kolumn

# usuwamy kolumny nieprzydatne do modelu
cols_to_drop = [
    "family",
    "origin",
    "climate",
    "insects",
    "tempmax.fahrenheit",
    "tempmin.fahrenheit",
    "latin_norm"
]

df = df.drop(columns=cols_to_drop, errors="ignore")

print("✅ Usunięto kolumny:", cols_to_drop)
print("📐 Aktualny rozmiar:", df.shape)

# Wypisanie wszystkich kolumn dokładnie tak, jak są zapisane w pliku
print("🔎 Lista pozostałych kolumn w dataframe:")
for col in df.columns:
    print(f"- {col}")

df.to_csv(EXPORT_DIR / "plants_clean2.csv", index=False)
df = pd.read_csv(EXPORT_DIR / "plants_clean2.csv")



✅ Usunięto kolumny: ['family', 'origin', 'climate', 'insects', 'tempmax.fahrenheit', 'tempmin.fahrenheit', 'latin_norm']
📐 Aktualny rozmiar: (159, 10)
🔎 Lista pozostałych kolumn w dataframe:
- latin
- category
- ideallight
- toleratedlight
- watering
- use
- tempmax.celsius
- tempmin.celsius
- toxicity_animals
- toxicity_humans


In [26]:
# podgląd aktualnych danych po czyszczeniu

print("\n🔎 Typy danych:")
print(df.dtypes)

print("\n🔎 Pierwsze 5 wierszy:")
display(df.head())




🔎 Typy danych:
latin               object
category            object
ideallight          object
toleratedlight      object
watering            object
use                 object
tempmax.celsius      int64
tempmin.celsius      int64
toxicity_animals    object
toxicity_humans     object
dtype: object

🔎 Pierwsze 5 wierszy:


,latin,category,ideallight,toleratedlight,watering,use,tempmax.celsius,tempmin.celsius,toxicity_animals,toxicity_humans
0,Aeschynanthus lobianus,Hanging,Bright light,Direct sunlight,Keep moist between watering. Can be a bit dry ...,Hanging; Flower; Tertiary,32,14,nietoksyczna,nietoksyczna
1,Adiantum raddianum,Fern,Bright light,Diffused,Keep moist between watering. Must not be dry b...,Potted plant; Ground cover; Table top,30,12,nietoksyczna,nietoksyczna
2,Aechmea fatsiata,Bromeliad,Bright light,Diffused,Water when soil is half dry. Change water in t...,Flower; Table top; Tertiary,30,12,nietoksyczna,może powodować lekkie podrażnienia
3,Agave angustilolia Marginata,Cactus And Succulent,6 or more hours of direct sunlight per day.,Direct sunlight.,Water only when the soil is dry. Must be dry b...,Potted plant; Primary; Secondary,35,5,"toksyczna (podrażnienia, wymioty)","toksyczna (sok drażni skórę, oczy)"
4,Aechmea ramosa,Bromeliad,Bright light,Diffused,Water when soil is half dry. Change water in t...,Flower; Table top; Primary,30,12,nietoksyczna,może powodować lekkie podrażnienia


In [27]:
# czy teraz pojawiły sie jakieś duplikaty wierszy?

# znajdowanie duplikatów w zbiorze

# liczba duplikatów
dup_count = df.duplicated().sum()
print(f"🔎 Liczba duplikatów w całym zbiorze: {dup_count}")

# jeśli są, podejrzyj kilka
if dup_count > 0:
    print("\n🔎 Przykładowe duplikaty:")
    display(df[df.duplicated()].head())


🔎 Liczba duplikatów w całym zbiorze: 0


In [28]:
# usunięcie duplikatów

before = df.shape[0]
df = df.drop_duplicates(keep="first").reset_index(drop=True)
after = df.shape[0]

print(f"✅ Usunięto {before - after} duplikatów")
print("📐 Aktualny rozmiar danych:", df.shape)

df.to_csv(EXPORT_DIR / "plants_clean3.csv", index=False)
df = pd.read_csv(EXPORT_DIR / "plants_clean3.csv")



✅ Usunięto 0 duplikatów
📐 Aktualny rozmiar danych: (159, 10)


In [29]:
df.describe


<bound method NDFrame.describe of                             latin              category  \
0          Aeschynanthus lobianus               Hanging   
1              Adiantum raddianum                  Fern   
2                Aechmea fatsiata             Bromeliad   
3    Agave angustilolia Marginata  Cactus And Succulent   
4                  Aechmea ramosa             Bromeliad   
..                            ...                   ...   
154         Schefflera arboricola            Schefflera   
155                 Spathiphyllum         Spathiphyllum   
156            Strelitzia nicolai         Foliage plant   
157         Zamioculcas zamifolia         Foliage plant   
158            Yucca elephantipes              Dracaena   

                                      ideallight    toleratedlight  \
0                                   Bright light   Direct sunlight   
1                                   Bright light          Diffused   
2                                   Bright ligh

In [30]:
# sanity-check i EDA  

num = df.select_dtypes(include="number")
print("=== DESCRIBE (numeryczne) ===")
print(num.describe().T.round(3))



=== DESCRIBE (numeryczne) ===
                 count    mean   std   min   25%   50%   75%   max
tempmax.celsius  159.0  29.642  2.71  22.0  28.0  30.0  30.0  40.0
tempmin.celsius  159.0  11.082  3.78  -5.0  10.0  12.0  12.0  18.0


## Wnioski

Zakresy max 22–40°C, min −5 do 18°C, jest mieszanka roślin ciepło- i chłodnolubnych.

Mediany: max ≈30°C, min ≈12°C → większość roślin „pokojowych”.

Rozstęp ćwiartkowy wąski (max 28–30–30): dużo gatunków z podobnym temp. max.

Min = −5°C: są gatunki tolerujące mróz 


In [31]:
print("Rozmiar zbioru:", df.shape) 

Rozmiar zbioru: (159, 10)


In [32]:
for c in df.columns:
    print(c)

latin
category
ideallight
toleratedlight
watering
use
tempmax.celsius
tempmin.celsius
toxicity_animals
toxicity_humans


In [33]:

# value_counts dla kolumn kategorycznych
CATS = [
    "category","ideallight","toleratedlight","watering","use"
    "toxicity_animals","toxicity_humans",
    
]
cats_present = [c for c in CATS if c in df.columns]
for c in cats_present:
    print(f"\n=== {c} (value_counts) ===")
    vc = df[c].fillna("BRUK").value_counts(dropna=False)
    print(vc.head(20))




=== category (value_counts) ===
category
Foliage plant           18
Cactus And Succulent    15
Fern                    15
Palm                    15
Bromeliad               14
Hanging                 13
Other                    9
Flower                   9
Philodendron             8
Aralia                   8
Spathiphyllum            7
Ficus                    6
Dracaena                 5
Anthurium                4
Sansevieria              4
Schefflera               4
Dieffenbachia            3
Aglaonema                1
Grass                    1
Name: count, dtype: int64

=== ideallight (value_counts) ===
ideallight
Bright light                                   101
6 or more hours of direct sunlight per day.     57
Prefers bright, indirect sunlight.               1
Name: count, dtype: int64

=== toleratedlight (value_counts) ===
toleratedlight
Diffused            87
Direct sunlight.    53
/                   12
Direct sunlight      7
Name: count, dtype: int64

=== watering (value_c

## Wnioski i działanie

1. category: dużo pod-kategorii (Foliage, Fern, Palm, Dracaena itd.), należy zmniejszyć ilość kategorii do około 6

2. ideallight: 3 formy, z czego 1 skrajnie rzadka → ujednolić do: Niskie/Średnie/Wysokie

3. toleratedlight: "Diffused" (światło rozproszone)
"Direct sunlight." =  "Direct sunlight" (pełne słońce) oraz
12 braków jako '/', nalezy je uzupełnić informacjami z ideallight, resztę zmapować

3. watering: długie zdania → zrób 3 grupy (Rzadko/Umiarkowanie/często) po słowach-kluczach.

4. toxicity_*: długi słownik opisowy → sprowadzamy do binarnego Toksyczna/Nietoksyczna 



## Feature Engineering

In [35]:
# Dodanie pasm temperatur i flag tolerancji

# Pasy temperatur (polskie etykiety)
def tempmin_band_pl(x):
    if x <= 5:
        return "Odporna na chłód"
    if 6 <= x <= 10:
        return "Toleruje chłód"
    if 11 <= x <= 13:
        return "Tylko temperatura pokojowa"
    return "Lubi ciepło"

def tempmax_band_pl(x):
    if x < 28:
        return "Wrażliwa na upał"
    if 28 <= x <= 34:
        return "Normalna tolerancja ciepła"
    return "Odporna na upał"  # x >= 35

df["tempmin_pasmo"] = df["tempmin.celsius"].apply(tempmin_band_pl)
df["tempmax_pasmo"] = df["tempmax.celsius"].apply(tempmax_band_pl)

# flagi binarne (0/1)
df["is_cold_tolerant"] = (df["tempmin.celsius"] <= 5).astype(int)
df["is_heat_tolerant"] = (df["tempmax.celsius"] >= 35).astype(int)

# zapis z nowymi kolumnami
df.to_csv(EXPORT_DIR / "plants_clean4.csv", index=False)
df = pd.read_csv(EXPORT_DIR / "plants_clean4.csv")


df.shape

(159, 14)

In [36]:
# sprawdzenie ile wierszy ma błąd (tempmax < tempmin)

bad = (df["tempmax.celsius"] < df["tempmin.celsius"]).sum()
print("Liczba błędnych wierszy:", bad)

df.shape


Liczba błędnych wierszy: 0


(159, 14)

In [37]:
# słownik do mapowania kolumny category na 6 grup

category_map = {
    # Sukulenty i kaktusy
    "Cactus And Succulent": "Sukulent / kaktus",

    # Palmy i drzewka ozdobne
    "Palm": "Palma / drzewko ozdobne",
    "Ficus": "Palma / drzewko ozdobne",
    "Aralia": "Palma / drzewko ozdobne",
    "Schefflera": "Palma / drzewko ozdobne",

    # Paprocie i rośliny zielone
    "Fern": "Paproć / roślina zielona",
    "Foliage plant": "Paproć / roślina zielona",
    "Aglaonema": "Paproć / roślina zielona",
    "Dieffenbachia": "Paproć / roślina zielona",
    "Dracaena": "Paproć / roślina zielona",
    "Philodendron": "Paproć / roślina zielona",
    "Sansevieria": "Paproć / roślina zielona",
    "Grass": "Paproć / roślina zielona",

    # Rośliny kwitnące
    "Flower": "Roślina kwitnąca",
    "Anthurium": "Roślina kwitnąca",
    "Spathiphyllum": "Roślina kwitnąca",
    "Bromeliad": "Roślina kwitnąca",

    # Wiszące
    "Hanging": "Wisząca",

    # Topiairy i inne → Inne
    "Topiairy": "Inne",
    "Other": "Inne",
}

# mapowanie kolumny
df["category_group"] = df["category"].map(category_map).fillna("Inne")

# sprawdzenie rozkładu
print(df["category_group"].value_counts())





category_group
Paproć / roślina zielona    55
Roślina kwitnąca            34
Palma / drzewko ozdobne     33
Sukulent / kaktus           15
Wisząca                     13
Inne                         9
Name: count, dtype: int64


In [38]:
df.to_csv(EXPORT_DIR / "plants_clean5.csv", index=False)
df = pd.read_csv(EXPORT_DIR / "plants_clean5.csv")


df.shape

(159, 15)

In [21]:
df = pd.read_csv("plants_clean5.csv")

# sprawdzmy jakie wartości unikalne zostały w ideallight po normalizacji
print(df["ideallight"].unique())

# pokażmy wszystkie rekordy gdzie w ideallight jest "shade" / "low"
mask_shade = df["ideallight"].str.contains("shade|low", case=False, na=False)
print(df.loc[mask_shade, ["latin", "ideallight"]])

df.value_counts


['Bright light' '6 or more hours of direct sunlight per day.'
 'Prefers bright, indirect sunlight.']
Empty DataFrame
Columns: [latin, ideallight]
Index: []


<bound method DataFrame.value_counts of                             latin              category  \
0          Aeschynanthus lobianus               Hanging   
1              Adiantum raddianum                  Fern   
2                Aechmea fatsiata             Bromeliad   
3    Agave angustilolia Marginata  Cactus And Succulent   
4                  Aechmea ramosa             Bromeliad   
..                            ...                   ...   
154         Schefflera arboricola            Schefflera   
155                 Spathiphyllum         Spathiphyllum   
156            Strelitzia nicolai         Foliage plant   
157         Zamioculcas zamifolia         Foliage plant   
158            Yucca elephantipes              Dracaena   

                                      ideallight    toleratedlight  \
0                                   Bright light   Direct sunlight   
1                                   Bright light          Diffused   
2                                   Brigh

In [39]:
# Policz wystąpienia każdej wartości w kolumnie "ideallight" i "toleratedlight"
print("\nLiczebność poszczególnych wartości:")
print(df["ideallight"].value_counts())

print(df["toleratedlight"].value_counts())


Liczebność poszczególnych wartości:
ideallight
Bright light                                   101
6 or more hours of direct sunlight per day.     57
Prefers bright, indirect sunlight.               1
Name: count, dtype: int64
toleratedlight
Diffused            87
Direct sunlight.    53
/                   12
Direct sunlight      7
Name: count, dtype: int64


In [40]:
# unikalne wartości w toleratedlight
print(df["toleratedlight"].unique())

# wyszukaj wpisy z "shade" / "low" w toleratedlight
mask = df["toleratedlight"].str.contains("shade|low", case=False, na=False)
print(df.loc[mask, ["latin", "toleratedlight"]])


['Direct sunlight' 'Diffused' 'Direct sunlight.' '/']
Empty DataFrame
Columns: [latin, toleratedlight]
Index: []


In [41]:
import re

def norm(s):
    if pd.isna(s): return s
    return re.sub(r"\.+$", "", str(s)).strip().lower()

ideal = df["ideallight"].map(norm)
tolerated = df["toleratedlight"].map(norm)

def classify_light(ideal, tolerated):
    # priorytet: ideallight
    text = str(ideal) if pd.notna(ideal) else str(tolerated)
    text = text.lower()

    if any(x in text for x in ["low", "shaded"]):
        return "Cień"
    if any(x in text for x in ["direct sunlight", "full sun", "6 or more hours"]):
        return "Bezpośrednie światło"
    if any(x in text for x in ["bright", "indirect", "diffused"]):
        return "Rozproszone światło"
    return np.nan

df["light_level_clean"] = [
    classify_light(i, t) for i, t in zip(ideal, tolerated)
]

print(df["light_level_clean"].value_counts(dropna=False))

df.to_csv(EXPORT_DIR / "plants_clean6.csv", index=False)
df = pd.read_csv(EXPORT_DIR / "plants_clean6.csv")


light_level_clean
Rozproszone światło     101
Bezpośrednie światło     58
Name: count, dtype: int64


In [42]:
# ile wierszy ma identyczne wartości w ideallight i toleratedlight
same = (df["ideallight"].fillna("") == df["toleratedlight"].fillna("")).sum()
print("Identyczne wpisy:", same, "z", len(df))

# lista wszystkich kolumn
print("Kolumny w pliku:")
print(df.columns.tolist())

# liczba kolumn
print("\nLiczba kolumn:", len(df.columns))

# szybki podgląd typów danych
print("\nTypy danych:")
print(df.dtypes)

df.shape

Identyczne wpisy: 0 z 159
Kolumny w pliku:
['latin', 'category', 'ideallight', 'toleratedlight', 'watering', 'use', 'tempmax.celsius', 'tempmin.celsius', 'toxicity_animals', 'toxicity_humans', 'tempmin_pasmo', 'tempmax_pasmo', 'is_cold_tolerant', 'is_heat_tolerant', 'category_group', 'light_level_clean']

Liczba kolumn: 16

Typy danych:
latin                object
category             object
ideallight           object
toleratedlight       object
watering             object
use                  object
tempmax.celsius       int64
tempmin.celsius       int64
toxicity_animals     object
toxicity_humans      object
tempmin_pasmo        object
tempmax_pasmo        object
is_cold_tolerant      int64
is_heat_tolerant      int64
category_group       object
light_level_clean    object
dtype: object


(159, 16)

In [43]:
# unikalne wartości
print(df["watering"].unique())

# albo zliczone wartości
print(df["watering"].value_counts(dropna=False))


['Keep moist between watering. Can be a bit dry between watering'
 'Keep moist between watering. Must not be dry between watering'
 'Water when soil is half dry. Change water in the vase regularly.'
 'Water only when the soil is dry. Must be dry between watering'
 'Keep moist between watering. Water when soil is half dry.'
 'Water when soil is half dry. Can be dry between watering.'
 'Water only when dry or when soil is half dry.'
 'Keep moist between watering. Can dry between watering'
 'Can be dry between watering. Water when soil is half dry.'
 'Water only when dry. Must be dry between watering'
 'Must be dry between watering. Water only when dry.'
 'Change water regularly in the vase. Water when soil is half dry.']
watering
Keep moist between watering. Water when soil is half dry.           64
Keep moist between watering. Must not be dry between watering       37
Water when soil is half dry. Can be dry between watering.           18
Water only when the soil is dry. Must be dry betw

In [44]:
import pandas as pd

# słownik mapowania po pełnych tekstach
map_watering = {
    "Keep moist between watering. Water when soil is half dry.": "Umiarkowanie",
    "Keep moist between watering. Must not be dry between watering": "Często",
    "Water when soil is half dry. Can be dry between watering.": "Umiarkowanie",
    "Keep moist between watering. Can dry between watering": "Umiarkowanie",
    "Water only when the soil is dry. Must be dry between watering": "Rzadko",
    "Change water regularly in the vase. Water when soil is half dry.": "Umiarkowanie",
    "Water when soil is half dry. Change water in the vase regularly.": "Umiarkowanie",
    "Water only when dry or when soil is half dry.": "Umiarkowanie",
    "Keep moist between watering. Can be a bit dry between watering": "Umiarkowanie",
    "Can be dry between watering. Water when soil is half dry.": "Umiarkowanie",
    "Water only when dry. Must be dry between watering": "Rzadko",
    "Must be dry between watering. Water only when dry.": "Rzadko"
}

# nowa kolumna z grupą
df["watering_group"] = df["watering"].map(map_watering)

# rozkład po grupach
print(df["watering_group"].value_counts(dropna=False))

# zapis do pliku
df.to_csv("plants_clean7.csv", index=False)

watering_group
Umiarkowanie    108
Często           37
Rzadko           14
Name: count, dtype: int64


In [45]:
df.shape

(159, 17)

In [46]:
df.to_csv(EXPORT_DIR / "plants_clean7.csv", index=False)
df = pd.read_csv(EXPORT_DIR / "plants_clean7.csv")


# uproszczona binarka: 1 = Toksyczna, 0 = Nietoksyczna
def is_toxic(human, animal):
    # jeśli w obu opisach jest słowo 'nietoksyczna' → 0
    if pd.notna(human) and "nietoksycz" in human.lower() and \
       pd.notna(animal) and "nietoksycz" in animal.lower():
        return 0
    # w każdym innym przypadku uznajemy za toksyczną
    return 1

df["toxicity_any"] = df.apply(lambda row: is_toxic(row["toxicity_humans"], row["toxicity_animals"]), axis=1)

print(df["toxicity_any"].value_counts())
print(df[["latin", "toxicity_humans", "toxicity_animals", "toxicity_any"]].head(10))

df.to_csv("plants_clean8.csv", index=False)

df.shape

toxicity_any
1    90
0    69
Name: count, dtype: int64
                          latin  \
0        Aeschynanthus lobianus   
1            Adiantum raddianum   
2              Aechmea fatsiata   
3  Agave angustilolia Marginata   
4                Aechmea ramosa   
5              Aechmea fasciata   
6                Agave filifera   
7           Adiantum hispidulum   
8               Agave attenuata   
9                     Aglaonema   

                                     toxicity_humans  \
0                                       nietoksyczna   
1                                       nietoksyczna   
2                 może powodować lekkie podrażnienia   
3                 toksyczna (sok drażni skórę, oczy)   
4                 może powodować lekkie podrażnienia   
5                 może powodować lekkie podrażnienia   
6  toksyczna (podrażnienia skóry, układu pokarmow...   
7                                       nietoksyczna   
8               toksyczna (sok drażniący, oparzenia)   

(159, 18)

In [50]:
cols = df.columns.tolist()
print("Kolumny:", cols)
print("Podejrzane (case-insensitive, z 'use'):", [c for c in cols if "use" in c.lower()])


Kolumny: ['latin', 'category', 'ideallight', 'toleratedlight', 'watering', 'tempmax.celsius', 'tempmin.celsius', 'toxicity_animals', 'toxicity_humans', 'tempmin_pasmo', 'tempmax_pasmo', 'is_cold_tolerant', 'is_heat_tolerant', 'category_group', 'light_level_clean', 'watering_group', 'toxicity_any']
Podejrzane (case-insensitive, z 'use'): []


In [52]:
df.to_csv(EXPORT_DIR / "plants_clean8.csv", index=False)
df = pd.read_csv(EXPORT_DIR / "plants_clean8.csv")


# usunięcie kolumny use

df = df.drop(columns=["use"], errors="ignore")


df.to_csv(NOTEBOOK_DIR.parent / "data" / "plants_clean_final.csv", index=False)

df.shape


(159, 17)

In [53]:
#wymień wszystkie kolumny wystepujące w df
print("Kolumny w pliku:")       
for col in df.columns:
    print(f"- {col}")   
            

Kolumny w pliku:
- latin
- category
- ideallight
- toleratedlight
- watering
- tempmax.celsius
- tempmin.celsius
- toxicity_animals
- toxicity_humans
- tempmin_pasmo
- tempmax_pasmo
- is_cold_tolerant
- is_heat_tolerant
- category_group
- light_level_clean
- watering_group
- toxicity_any


In [54]:
import pandas as pd
import json
from pathlib import Path

# --- Ścieżki ---
NOTEBOOK_DIR = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
DATA_PATH = NOTEBOOK_DIR.parent / "data" / "plants_clean_final.csv"
ARTIFACTS_DIR = NOTEBOOK_DIR.parent / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

OUT_JSON = ARTIFACTS_DIR / "feature_list.json"

# --- Wczytaj dane ---
df = pd.read_csv(DATA_PATH)
print("✅ Wczytano:", DATA_PATH)
print("🔢 Wiersze x kolumny:", df.shape)
# lista cech do modelu
FEATURES = [
    {"name": "tempmin_pasmo",     "type": "categorical", "encoding": "onehot"},
    {"name": "tempmax_pasmo",     "type": "categorical", "encoding": "onehot"},
    {"name": "is_cold_tolerant",  "type": "numeric"},
    {"name": "is_heat_tolerant",  "type": "numeric"},
    {"name": "category_group",    "type": "categorical", "encoding": "onehot"},
    {"name": "light_level_clean", "type": "categorical", "encoding": "onehot"},
    {"name": "watering_group",    "type": "categorical", "encoding": "onehot"},
    {"name": "toxicity_any",      "type": "numeric"}
]

# walidacja – czy wszystkie kolumny istnieją w CSV
missing = [f["name"] for f in FEATURES if f["name"] not in df.columns]
if missing:
    print("⚠️ Brakuje kolumn:", missing)

# zbuduj strukturę JSON
feature_list = {
    "features_order": FEATURES,
    "target": None,
    "notes": "Cechy użyte do modelu rekomendacji roślin; wszystkie kategoryczne kodowane One-Hot."
}

# --- Zapis do artifacts ---
OUT_JSON.write_text(json.dumps(feature_list, ensure_ascii=False, indent=2))
print(f"💾 Zapisano plik: {OUT_JSON}")

✅ Wczytano: c:\Users\annas\projekty_infoshare\projekt ML\data\plants_clean_final.csv
🔢 Wiersze x kolumny: (159, 17)
💾 Zapisano plik: c:\Users\annas\projekty_infoshare\projekt ML\artifacts\feature_list.json


## Co zrobiłam i po co

- Wczytałam `plants_clean_final.csv` i zdefiniowałam **listę cech wejściowych do modelu** (`feature_list.json`).
- W pliku JSON zapisałam **kolejność cech** oraz **sposób ich kodowania**
- **Nie dodaję** kolumn opisowych (np. pełne teksty toksyczności, łacińska nazwa) – zostają w CSV, ale **nie trafiają do modelu**.
- Ten plik będzie użyty w kolejnym kroku do **automatycznego zbudowania pipeline’u** (OHE → skalowanie → PCA → klastrowanie) w pliku 01-Feature_Transformation.ipynb


In [55]:
df.shape

(159, 17)